# Chapter 4 — Separate What From How

**Book alignment:** DSPy From First Principles, Chapter 4

**Question this notebook isolates:** Under a frozen contract, frozen cases, and frozen metrics, does explicit reasoning (ChainOfThought vs Predict) change rewrite quality — and does the metric change the answer?


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy
from common.data import teaching_cases
from common.dspy_program import RewriteSentence
from common.metrics import editorial_metric_v1, editorial_metric_v2


## One contract, two policies

Both strategies wrap the same signature and receive the same values (the two Chapter 5 analysis fields pass as empty strings to both). The recorded `ed-025` pair below stands in for the measured outputs: direct prediction shifts the deadline trigger from delivery to receipt; the reasoned rewrite preserves it. Modules are constructed, never executed.


In [ ]:
direct = dspy.Predict(RewriteSentence)
reasoned = dspy.ChainOfThought(RewriteSentence)

ed025 = {c.case_id: c for c in teaching_cases()}["ed-025"]
PREDICT_TEXT = "You must notify the sender in writing about the defect within 30 days of receiving the item."
COT_TEXT = "You must notify in writing about the defect within 30 days of delivery."

print("source: ", ed025.sentence)
print("Predict:", PREDICT_TEXT)
print("CoT:    ", COT_TEXT)


In [ ]:
assert set(direct.signature.input_fields) == set(reasoned.predict.signature.input_fields)
assert set(direct.signature.output_fields) == set(RewriteSentence.output_fields)
assert "delivery" in COT_TEXT and "receiving the item" in PREDICT_TEXT
print("only deliberate intervention: the execution policy")


## Two metrics, two answers

The structural metric rates both rewrites highly. A deterministic fixture judge — preserved on every constraint for the reasoned rewrite, violated on the delivery-trigger constraint for the direct one — reproduces the book's v2 finding: the whole aggregate gap reduces to this one case's arithmetic.


In [ ]:
class FixtureJudge:
    """Deterministic stand-in for the validated semantic judge (no LM call)."""

    def __init__(self, violate_last: bool) -> None:
        self.violate_last = violate_last

    def judge(self, original_sentence, candidate_rewrite, editorial_goal, constraints):
        verdicts = []
        for pos, constraint in enumerate(constraints):
            verdict = "violated" if (self.violate_last and pos == len(constraints) - 1) else "preserved"
            verdicts.append(_Verdict(constraint, verdict))
        preserved = sum(v.verdict == "preserved" for v in verdicts)
        violated = sum(v.verdict == "violated" for v in verdicts)
        return _Report(tuple(verdicts), preserved, violated)


class _Verdict:
    def __init__(self, constraint, verdict):
        self.constraint = constraint
        self.verdict = verdict

    def to_record(self):
        return {"constraint": self.constraint, "verdict": self.verdict}


class _Report:
    def __init__(self, constraints, preserved, violated):
        self.constraints = constraints
        self.preserved = preserved
        self.violated = violated
        self.unclear = 0
        self.parse_failures = 0
        self.semantic_factor = preserved / max(len(constraints), 1)
        self.provenance = {"judge": "deterministic-fixture"}


v1_predict = editorial_metric_v1(ed025, PREDICT_TEXT).score
v1_cot = editorial_metric_v1(ed025, COT_TEXT).score
v2_predict = editorial_metric_v2(ed025, PREDICT_TEXT, judge=FixtureJudge(violate_last=True)).score
v2_cot = editorial_metric_v2(ed025, COT_TEXT, judge=FixtureJudge(violate_last=False)).score

print(f"v1: Predict {v1_predict:.3f} vs CoT {v1_cot:.3f}")
print(f"v2: Predict {v2_predict:.3f} vs CoT {v2_cot:.3f}")
print(f"case swing v1->v2 on Predict: {v1_predict - v2_predict:.3f} spread over 37 cases")


In [ ]:
assert round(v1_predict, 3) == 0.933
assert round(v1_cot, 3) == 0.967
assert v2_predict == 0.30
assert v2_cot == v1_cot
# Book arithmetic: one 0.633 case swing explains the v1-vs-v2 aggregate gap.
case_swing = v1_predict - v2_predict
assert abs(case_swing / 37 - (0.0313 - 0.0141)) < 1e-3
print("v1 rates a semantically invalid rewrite very highly; v2 caps it at 0.30")


## Reasoning has a price

Recorded totals from the 222-call comparison: reasoning costs latency and completion tokens. Whether that price is worth paying depends on the application's cost of the failure it avoids — not on the aggregate delta alone.


In [ ]:
COST = {
    "predict_latency": 1.879,
    "cot_latency": 3.113,
    "predict_completion": 8490,
    "cot_completion": 14481,
    "predict_total": 46971,
    "cot_total": 55959,
    "hard_gate_failures_each": 6,
}
latency_ratio = COST["cot_latency"] / COST["predict_latency"]
completion_ratio = COST["cot_completion"] / COST["predict_completion"]
total_ratio = COST["cot_total"] / COST["predict_total"]
print(f"latency {latency_ratio:.2f}x, completion tokens {completion_ratio:.2f}x, total tokens {total_ratio:.2f}x")


In [ ]:
assert 1.6 < latency_ratio < 1.7
assert 1.7 < completion_ratio < 1.75
assert 1.15 < total_ratio < 1.25
assert COST["hard_gate_failures_each"] == 6
print("66% more time, 71% more completion tokens, identical gate failures")


## What we earned

Holding the contract fixed makes the strategy comparison legible: a small lexical gain near run-to-run movement under v1, and one inspectable semantic save under v2 that v1 could not see. The measurement decided the answer — a warning before handing the metric to optimizers.

Notebook 05 / Chapter 5 changes the architecture instead of the policy: real systems are several calls with Python between them, so which intermediate stages actually influence the result?
